In [1]:
!pip install -U ecmwf-opendata xarray cfgrib netCDF4 pandas eccodes

  Using cached xarray-2026.7.0-py3-none-any.whl.metadata (12 kB)
Using cached xarray-2026.7.0-py3-none-any.whl (1.4 MB)
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.8 MB 5.5 MB/s eta 0:00:02
   --------- ------------------------------ 2.4/9.8 MB 6.9 MB/s eta 0:00:02
   ---------------- ----------------------- 3.9/9.8 MB 7.1 MB/s eta 0:00:01
   ------------------------ --------------- 6.0/9.8 MB 7.7 MB/s eta 0:00:01
   -------------------------------- ------- 7.9/9.8 MB 8.0 MB/s eta 0:00:01
   -------------------------------------- - 9.4/9.8 MB 7.9 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 7.7 MB/s  0:00:01

  Attempting uninstall: pandas

    Found existing installation: pandas 2.3.3

   ---------------------------------------- 0/4 [pandas]
   ---------------------------------------- 0/4 [pandas]
   ---------------------------------------- 0/4 [pandas]
   ----------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires pandas<3,>=1.4.0, but you have pandas 3.0.5 which is incompatible.


In [2]:
import xarray as xr
import pandas as pd

from pathlib import Path
from ecmwf.opendata import Client

In [3]:
FEATURES = [
    "temperature",
    "pressure"
]

PARAMETERS = {
    "temperature": "2 m above ground",
    "pressure": "Mean Sea Level Pressure"
}

LATITUDE = (5, 35)
LONGITUDE = (65, 100)

FORECAST_STEP = 0

OUTPUT_DIR = Path("ecmwf_output")
OUTPUT_DIR.mkdir(exist_ok=True)

In [4]:
FEATURE_MAP = {

    "temperature": {
        "param": "2t",
        "shortName": "2t"
    },

    "pressure": {
        "param": "msl",
        "shortName": "msl"
    }
}

In [5]:
def validate_features():

    for feature in FEATURES:

        if feature not in FEATURE_MAP:
            raise ValueError(
                f"Unsupported feature: {feature}"
            )

    print("Features validated successfully!")

In [6]:
validate_features()

Features validated successfully!


In [7]:
client = Client(
    source="ecmwf",
    model="ifs",
    resol="0p25"
)

print("ECMWF client ready!")

To ensure the stability of our systems and to preserve resources for our operational activities (network, compute, etc.), access to the open-data portal is limited to 500 simultaneous connections. This limit helps us guarantee reliable service for our operational users, especially during periods of high demand. For added reliability, the open-data is replicated across AWS, Azure, and Google Cloud. If you experience difficulties accessing the portal directly, you can also retrieve the data from these cloud platforms.


ECMWF client ready!


In [8]:
def download_ecmwf():

    output_file = OUTPUT_DIR / "ecmwf_data.grib2"

    params = []

    for feature in FEATURES:

        params.append(
            FEATURE_MAP[feature]["param"]
        )

    print("Downloading ECMWF data...")
    print("Parameters:", params)
    print("Forecast step:", FORECAST_STEP)

    result = client.retrieve(
        type="fc",
        step=FORECAST_STEP,
        param=params,
        target=str(output_file)
    )

    print("\nDownload completed!")
    print("Forecast datetime:", result.datetime)
    print("Saved at:", output_file)

    return output_file, result.datetime

In [9]:
grib_file, forecast_time = download_ecmwf()

Parameters: ['2t', 'msl']
Forecast step: 0


20260910120000-0h-oper-fc.grib2:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

By downloading data from the ECMWF open data dataset, you agree to the terms: Attribution 4.0 International (CC BY 4.0). Please attribute ECMWF when downloading this data.

Download completed!
Forecast datetime: 2026-09-10 12:00:00
Saved at: ecmwf_output\ecmwf_data.grib2


In [10]:
temperature = xr.open_dataset(
    grib_file,
    engine="cfgrib",
    backend_kwargs={
        "filter_by_keys": {
            "shortName": "2t"
        }
    }
)

print("Temperature dataset opened!")
print(temperature.data_vars)

Temperature dataset opened!
Data variables:
    t2m      (latitude, longitude) float32 4MB ...


In [11]:
pressure = xr.open_dataset(
    grib_file,
    engine="cfgrib",
    backend_kwargs={
        "filter_by_keys": {
            "shortName": "msl"
        }
    }
)

print("Pressure dataset opened!")
print(pressure.data_vars)

Pressure dataset opened!
Data variables:
    msl      (latitude, longitude) float32 4MB ...


In [12]:
def crop_region(dataset):

    lat_min, lat_max = LATITUDE
    lon_min, lon_max = LONGITUDE

    lat_values = dataset.latitude.values

    if lat_values[0] > lat_values[-1]:

        dataset = dataset.sel(
            latitude=slice(lat_max, lat_min)
        )

    else:

        dataset = dataset.sel(
            latitude=slice(lat_min, lat_max)
        )

    dataset = dataset.sel(
        longitude=slice(lon_min, lon_max)
    )

    return dataset

In [13]:
temperature = crop_region(temperature)

In [14]:
pressure = crop_region(pressure)

In [15]:
temperature_c = temperature["t2m"] - 273.15

pressure_hpa = pressure["msl"] / 100

In [16]:
final_dataset = xr.Dataset(
    {
        "temperature_2m_C": temperature_c,
        "pressure_msl_hPa": pressure_hpa
    }
)

print("Final ECMWF dataset created!")
print(final_dataset)

Final ECMWF dataset created!
<xarray.Dataset> Size: 139kB
Dimensions:            (latitude: 121, longitude: 141)
Coordinates:
  * latitude           (latitude) float64 968B 35.0 34.75 34.5 ... 5.5 5.25 5.0
  * longitude          (longitude) float64 1kB 65.0 65.25 65.5 ... 99.75 100.0
    time               datetime64[ns] 8B 2026-09-10T12:00:00
    step               timedelta64[ns] 8B 00:00:00
    heightAboveGround  float64 8B ...
    valid_time         datetime64[ns] 8B 2026-09-10T12:00:00
    meanSea            float64 8B ...
Data variables:
    temperature_2m_C   (latitude, longitude) float32 68kB 23.35 24.35 ... 30.69
    pressure_msl_hPa   (latitude, longitude) float32 68kB 1.012e+03 ... 1.009...


In [17]:
df = final_dataset.to_dataframe().reset_index()

df = df[
    [
        "time",
        "latitude",
        "longitude",
        "temperature_2m_C",
        "pressure_msl_hPa"
    ]
]

print("DataFrame created!")
print("Shape:", df.shape)

display(df.head(10))

DataFrame created!
Shape: (17061, 5)


,time,latitude,longitude,temperature_2m_C,pressure_msl_hPa
0,2026-09-10 12:00:00,35.0,65.00,23.348236,1012.475647
1,2026-09-10 12:00:00,35.0,65.25,24.348236,1011.635620
2,2026-09-10 12:00:00,35.0,65.50,24.941986,1010.875610
3,2026-09-10 12:00:00,35.0,65.75,22.129486,1012.235596
4,2026-09-10 12:00:00,35.0,66.00,18.254486,1014.315613
5,2026-09-10 12:00:00,35.0,66.25,20.098236,1012.915649
6,2026-09-10 12:00:00,35.0,66.50,20.910736,1012.155640
7,2026-09-10 12:00:00,35.0,66.75,17.348236,1014.675598
8,2026-09-10 12:00:00,35.0,67.00,16.316986,1014.835632
9,2026-09-10 12:00:00,35.0,67.25,16.785736,1014.995605


In [18]:
csv_file = OUTPUT_DIR / "ecmwf_weather_data.csv"

df.to_csv(
    csv_file,
    index=False
)

print("CSV saved successfully!")
print("File:", csv_file)

CSV saved successfully!
File: ecmwf_output\ecmwf_weather_data.csv


In [19]:
netcdf_file = OUTPUT_DIR / "ecmwf_weather_data.nc"

final_dataset.to_netcdf(
    netcdf_file
)

print("NetCDF saved successfully!")
print("File:", netcdf_file)

NetCDF saved successfully!
File: ecmwf_output\ecmwf_weather_data.nc


In [20]:
print("Dataset Shape:")
print(df.shape)

print("\nFirst 10 rows:")
display(df.head(10))

print("\nMissing Values:")
print(df.isna().sum())

print("\nStatistics:")
display(
    df[
        [
            "temperature_2m_C",
            "pressure_msl_hPa"
        ]
    ].describe()
)

Dataset Shape:
(17061, 5)

First 10 rows:


,time,latitude,longitude,temperature_2m_C,pressure_msl_hPa
0,2026-09-10 12:00:00,35.0,65.00,23.348236,1012.475647
1,2026-09-10 12:00:00,35.0,65.25,24.348236,1011.635620
2,2026-09-10 12:00:00,35.0,65.50,24.941986,1010.875610
3,2026-09-10 12:00:00,35.0,65.75,22.129486,1012.235596
4,2026-09-10 12:00:00,35.0,66.00,18.254486,1014.315613
5,2026-09-10 12:00:00,35.0,66.25,20.098236,1012.915649
6,2026-09-10 12:00:00,35.0,66.50,20.910736,1012.155640
7,2026-09-10 12:00:00,35.0,66.75,17.348236,1014.675598
8,2026-09-10 12:00:00,35.0,67.00,16.316986,1014.835632
9,2026-09-10 12:00:00,35.0,67.25,16.785736,1014.995605



Missing Values:
time                0
latitude            0
longitude           0
temperature_2m_C    0
pressure_msl_hPa    0
dtype: int64

Statistics:


,temperature_2m_C,pressure_msl_hPa
count,17061.000000,17061.000000
mean,25.673016,1008.717407
std,8.482406,4.410000
min,-1.995514,1001.515625
25%,25.973236,1005.595642
50%,28.660736,1007.995605
75%,29.691986,1010.235596
max,41.316986,1026.755615
